# Comparação das três configurações

Notebook de análise de artefatos: não inicia downloads ou treinamento. Não substitui revisão humana de fundamentação.

## Comparação congelada
A: base sem documentos. B: mesma base com busca. C: mesma base e busca com adaptador selecionado. Todos usam geração gulosa, o mesmo esquema e até 768 tokens. Respostas brutas, prompts e trechos recuperados estão em `avaliacao/respostas/`. Sintaxe, esquema, referências e classificação de status são verificados separadamente. A fundamentação e a alucinação precisam de revisão independente; os valores automáticos não são acurácia semântica.

In [1]:
import json
from pathlib import Path
p=Path('avaliacao/placar.json')
results=json.loads(p.read_text()) if p.exists() else []
for m in results:
 print({k:m[k] for k in ['mode','n','json_syntax_valid_rate','schema_valid_rate','valid_json_and_refs_rate','status_match_rate','median_seconds','p95_seconds','semantic_correctness']})
if not results: print('Experimentos finais pendentes.')

{'mode': 'base_sem_contexto', 'n': 120, 'json_syntax_valid_rate': 1.0, 'schema_valid_rate': 0.4583333333333333, 'valid_json_and_refs_rate': 0.4583333333333333, 'status_match_rate': 0.23333333333333334, 'median_seconds': 2.5442166660013754, 'p95_seconds': 3.5870213607500774, 'semantic_correctness': 'PENDENTE: status_match não é acerto semântico ou fundamentação.'}
{'mode': 'rag_simples', 'n': 120, 'json_syntax_valid_rate': 1.0, 'schema_valid_rate': 0.7666666666666667, 'valid_json_and_refs_rate': 0.7416666666666667, 'status_match_rate': 0.49166666666666664, 'median_seconds': 3.0074570835004124, 'p95_seconds': 4.0999155961511855, 'semantic_correctness': 'PENDENTE: status_match não é acerto semântico ou fundamentação.'}
{'mode': 'rag_finetuning', 'n': 120, 'json_syntax_valid_rate': 0.9833333333333333, 'schema_valid_rate': 0.9333333333333333, 'valid_json_and_refs_rate': 0.9333333333333333, 'status_match_rate': 0.65, 'median_seconds': 2.466634229500414, 'p95_seconds': 2.7873066652007763, 'se

In [2]:
p=Path('avaliacao/cinco_erros.json')
print(json.dumps(json.loads(p.read_text()),ensure_ascii=False,indent=2) if p.exists() else 'Erros reais ainda não extraídos.')

[
  {
    "case_id": "test-respondivel-00-v01",
    "run_id": "final-A",
    "mode": "base_sem_contexto",
    "raw": "{\n  \"status\": \"informacao_insuficiente\",\n  \"resumo\": \"A ocorrência descreve um ponto de luz apagado junto a um ponto de ônibus à noite, mas não fornece detalhes suficientes para formular conclusões ou hipóteses.\",\n  \"hipoteses\": [],\n  \"informacoes_faltantes\": [\"local exato da ocorrência\", \"tempo exato em que o problema foi notado\", \"condições ambientais no momento do ocorrido\", \"informações sobre a manutenção recente da iluminação\"],\n  \"encaminhamento\": \"Para obter uma resposta mais precisa, é necessário fornecer mais detalhes sobre o local, o horário exato e quaisquer condições ambientais relevantes.\",\n  \"fontes\": []\n}",
    "expected_status": "analise_preliminar",
    "input": {
      "ocorrencia_id": "test-respondivel-00-v01",
      "pergunta": "Que conclusão é sustentada e o que falta?",
      "simulada": true,
      "relato": "Ponto

In [3]:
for m in results:
 print(m['mode'], 'recuperação respondível:',m.get('retrieval_respondible'))
 print('Por categoria:',json.dumps(m['by_category'],ensure_ascii=False,indent=2))
print('Revisão independente: preencher runs/final-*/revisao_humana.csv e então usar citizentech aggregate-review --help. Não preencher resultados sem revisar cada resposta.')

base_sem_contexto recuperação respondível: {'recall@1': 0.0, 'recall@3': 0.0}
Por categoria: {
  "respondivel": {
    "n": 40,
    "status_match_rate": 0.2
  },
  "nao_consta": {
    "n": 20,
    "status_match_rate": 0.05
  },
  "insuficiente": {
    "n": 20,
    "status_match_rate": 0.35
  },
  "inducao": {
    "n": 10,
    "status_match_rate": 0.1
  },
  "contradicao": {
    "n": 15,
    "status_match_rate": 0.6
  },
  "fora_escopo": {
    "n": 15,
    "status_match_rate": 0.13333333333333333
  }
}
rag_simples recuperação respondível: {'recall@1': 0.725, 'recall@3': 0.825}
Por categoria: {
  "contradicao": {
    "n": 15,
    "status_match_rate": 0.7333333333333333
  },
  "nao_consta": {
    "n": 20,
    "status_match_rate": 0.1
  },
  "insuficiente": {
    "n": 20,
    "status_match_rate": 0.65
  },
  "fora_escopo": {
    "n": 15,
    "status_match_rate": 0.4
  },
  "inducao": {
    "n": 10,
    "status_match_rate": 0.7
  },
  "respondivel": {
    "n": 40,
    "status_match_rate": 0.